[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch01-introduction/03-bai-hoc-dang-va-hieu-qua.ipynb)

# Bài học đắng và ba chiều của hiệu quả

Chương 1 kể lịch sử AI như một chuỗi điểm nghẽn được gỡ bỏ: logic, tri thức, đặc trưng, rồi hạ tầng. Nó dẫn
bài luận The Bitter Lesson (bài học đắng) của Richard Sutton: các phương pháp tổng quát biết tận dụng tính
toán rốt cuộc hiệu quả nhất. Rồi nó hỏi câu tiếp theo, câu của người làm hệ thống: nếu quy mô là đòn bẩy, thì
làm sao trả nổi cho quy mô?

Sổ tay này dựng cả hai nửa thành phép đo nhỏ trên MNIST, rồi tính các nhịp tăng trưởng mà chương đưa ra.

**Bạn sẽ làm:**
1. dựng lại điểm nghẽn thiết kế đặc trưng (feature engineering) thu nhỏ: 34 đặc trưng do người thiết kế,
   đặt cạnh điểm ảnh thô, theo lượng dữ liệu huấn luyện;
2. ước lượng độ co giãn của đường cong học tập theo ba hướng chi thêm 10 % phép toán: thêm dữ liệu, thêm lượt
   huấn luyện, hay mô hình lớn hơn;
3. đặt ba chiều hiệu quả lên phương trình chi phí của chương, mỗi chiều một phép đo trên CPU;
4. tính các nhịp 44.5×, 15 tháng và 3.4 tháng, và độ rộng của khoảng cách hệ thống.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn.functional as F
import torchvision
from scipy import ndimage

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định, như các sổ tay khác

tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
tap_kt = torchvision.datasets.MNIST(DATA, train=False, download=True)
anh_tho, y = tap.data[:10_000], tap.targets[:10_000].clone()           # 10,000 ảnh huấn luyện
anh_tho_kt, y_kt = tap_kt.data[:5_000], tap_kt.targets[:5_000].clone()  # 5,000 ảnh kiểm tra
X, X_kt = anh_tho.float().div(255).reshape(-1, 784), anh_tho_kt.float().div(255).reshape(-1, 784)
print(f"huấn luyện {len(X):,} ảnh · kiểm tra {len(X_kt):,} ảnh")

## 1. Điểm nghẽn thiết kế đặc trưng, thu nhỏ

Thời học thống kê, theo chương, các thuật toán như SVM (support vector machine) chỉ học được một cách bền vững sau khi con người đã
chuyển đầu vào thô thành các đặc trưng có cấu trúc. Học sâu thay điều đó bằng học đầu-cuối (end-to-end
learning): mạng học biểu diễn trực tiếp từ điểm ảnh.

Hàm dưới đây là phần việc của "chuyên gia". Nó đọc một chữ số viết tay như một người sẽ mô tả nó: bao nhiêu
mực, khung chữ cao hay rộng, có mấy lỗ kín (chữ số 8 có hai, chữ số 0 có một), một đường ngang cắt nét bao
nhiêu lần, mực dồn về góc nào, có đối xứng không. Tổng cộng 34 con số mỗi ảnh.

In [ ]:
def cat_qua(dong):
    """Số lần một đường quét đi từ nền vào nét."""
    return int(np.sum(np.diff(dong.astype(int)) == 1))


def dac_trung(anh):
    """34 đặc trưng do người thiết kế, cho mỗi ảnh 28 × 28."""
    ra = []
    for a in anh.numpy():
        net = a > 128
        dong, cot = np.nonzero(net)
        r0, r1, c0, c1 = dong.min(), dong.max(), cot.min(), cot.max()
        cao, rong = r1 - r0 + 1, c1 - c0 + 1
        khung = net[r0:r1 + 1, c0:c1 + 1]
        muc = khung.sum()
        _, so_vung = ndimage.label(~np.pad(khung, 1))     # vùng nền; trừ vùng bên ngoài thì còn số lỗ kín
        lo = so_vung - 1
        ngang = [cat_qua(khung[int(f * (cao - 1))]) for f in (0.25, 0.5, 0.75)]
        doc = [cat_qua(khung[:, int(f * (rong - 1))]) for f in (0.25, 0.5, 0.75)]
        nc, nr = cao // 2, rong // 2
        goc = [khung[:nc, :nr].sum() / muc, khung[:nc, nr:].sum() / muc,
               khung[nc:, :nr].sum() / muc, khung[nc:, nr:].sum() / muc]
        luoi = khung[np.ix_(np.arange(12) * cao // 12, np.arange(12) * rong // 12)].astype(float)
        vung = list(luoi.reshape(3, 4, 3, 4).mean((1, 3)).ravel())   # mật độ mực trên lưới 3 × 3
        ra.append([muc / 100, cao / 20, rong / 20, rong / cao,
                   (dong.mean() - r0) / cao, (cot.mean() - c0) / rong,
                   lo, lo == 1, lo == 2, *ngang, *doc, *goc,
                   (khung == khung[:, ::-1]).mean(), (khung == khung[::-1, :]).mean(),
                   khung[:nc].sum() / muc, khung[:, :nr].sum() / muc, ngang[1] >= 2, doc[1] >= 2, *vung])
    return torch.tensor(ra, dtype=torch.float32)


t0 = time.perf_counter()
D_tr, D_kt = dac_trung(anh_tho), dac_trung(anh_tho_kt)
do_tren_may(f"thời gian trích đặc trưng cho {len(D_tr) + len(D_kt):,} ảnh", time.perf_counter() - t0, "s")
tb, lech = D_tr.mean(0), D_tr.std(0) + 1e-6
D_tr, D_kt = (D_tr - tb) / lech, (D_kt - tb) / lech    # chuẩn hoá theo tập huấn luyện
assert D_tr.shape[1] == 34
print("số lỗ kín trung bình theo chữ số:",
      " ".join(f"{c}:{(D_tr[y == c, 6] * lech[6] + tb[6]).mean():.1f}" for c in range(10)))

Dòng in cuối là một phép thử nhanh cho đặc trưng "số lỗ kín": nếu nó làm đúng việc, chữ số 0, 6, 9 có gần một
lỗ, chữ số 8 gần hai, còn 1, 3, 5, 7 gần không.

Giờ là ba cách học, cùng một vòng lặp huấn luyện:

* đặc trưng thủ công cộng một bộ phân loại tuyến tính, đúng công thức của thời học thống kê;
* đặc trưng thủ công cộng một mạng nhỏ 34-64-10;
* điểm ảnh thô vào mạng 784-128-64-10, không ai nói cho nó biết lỗ kín là gì.

Mỗi cách được huấn luyện trên 30 tới 10,000 ảnh. Số bước cập nhật được giữ gần như nhau ở mọi cỡ dữ liệu (ít
nhất 3 epoch, và khoảng 30,000 lượt trình bày mẫu), để thứ thay đổi là lượng dữ liệu, không phải công sức tối ưu.

**Dự đoán:** với 100 ảnh, cách nào thắng? Với 10,000 ảnh thì sao?

In [ ]:
def mlp(rong, vao=784):
    lop = []
    for r in rong:
        lop += [torch.nn.Linear(vao, r), torch.nn.ReLU()]
        vao = r
    return torch.nn.Sequential(*lop, torch.nn.Linear(vao, 10))


def huan_luyen(kien_truc, X, y, so_epoch, toi_uu="adam", lr=None, batch=32, X_danh_gia=None, y_danh_gia=None):
    """Huấn luyện từ đầu với hạt giống cố định. Nếu có tập đánh giá, trả thêm độ chính xác sau mỗi epoch."""
    dat_hat_giong()
    mo_hinh = kien_truc()
    if toi_uu == "adam":
        bo_toi_uu = torch.optim.Adam(mo_hinh.parameters(), lr=lr or 1e-3)
    else:
        bo_toi_uu = torch.optim.SGD(mo_hinh.parameters(), lr=lr or 0.05)
    lich_su = []
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), batch):
            chon = thu_tu[i:i + batch]
            mat_mat = F.cross_entropy(mo_hinh(X[chon]), y[chon])
            bo_toi_uu.zero_grad()
            mat_mat.backward()
            bo_toi_uu.step()
        if X_danh_gia is not None:
            lich_su.append(do_chinh_xac(mo_hinh, X_danh_gia, y_danh_gia))
    return mo_hinh, lich_su


def do_chinh_xac(mo_hinh, X, y):
    with torch.no_grad():
        return (mo_hinh(X).argmax(1) == y).float().mean().item()


def so_tham_so(mo_hinh):
    return sum(p.numel() for p in mo_hinh.parameters())


CACH = {"đặc trưng thủ công + tuyến tính": (lambda: torch.nn.Linear(34, 10), D_tr, D_kt),
        "đặc trưng thủ công + mạng 34-64-10": (lambda: mlp((64,), vao=34), D_tr, D_kt),
        "điểm ảnh thô + mạng 784-128-64-10": (lambda: mlp((128, 64)), X, X_kt)}
CO_DU_LIEU = [30, 100, 300, 1_000, 3_000, 10_000]

t0 = time.perf_counter()
ket_qua = {ten: [] for ten in CACH}
for n in CO_DU_LIEU:
    so_epoch = max(3, 30_000 // n)
    for ten, (kien_truc, Dh, Dk) in CACH.items():
        m, _ = huan_luyen(kien_truc, Dh[:n], y[:n], so_epoch)
        ket_qua[ten].append(do_chinh_xac(m, Dk, y_kt))
do_tren_may("thời gian cho 18 lần huấn luyện", time.perf_counter() - t0, "s")
for ten, (kien_truc, _, _) in CACH.items():
    print(f"{ten}: {so_tham_so(kien_truc()):,} tham số")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for (ten, acc), mau, kieu in zip(ket_qua.items(), MAU, ("s--", "^:", "o-")):
    ax.plot(CO_DU_LIEU, 100 * np.array(acc), kieu, color=mau, lw=2, ms=6, label=ten)
ax.set_xscale("log")
ax.set_xticks(CO_DU_LIEU, [f"{n:,}" for n in CO_DU_LIEU])
ax.minorticks_off()
ax.set_xlabel("số ảnh huấn luyện (thang log)")
ax.set_ylabel("độ chính xác trên 5,000 ảnh kiểm tra (%)")
ax.set_title("Đặc trưng do người thiết kế và đặc trưng tự học (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()
for ten, acc in ket_qua.items():
    do_tren_may(f"{ten}, với {CO_DU_LIEU[-1]:,} ảnh", 100 * acc[-1], "%")

Đọc đồ thị theo hai câu hỏi: khi dữ liệu còn ít, đường nào ở trên? Và khi dữ liệu tăng, đường nào còn tiếp tục
đi lên?

Lý do đằng sau thì không phụ thuộc máy. Bộ phân loại tuyến tính chỉ có 350 tham số và chỉ nhìn thấy 34 con số
mà con người chọn, nên nó học được nhanh từ vài trăm ảnh, nhưng trần của nó là lượng thông tin con người đã cài
vào 34 con số đó. Mạng trên điểm ảnh thô có hơn một trăm nghìn tham số và phải tự tìm ra đặc trưng: nó cần
nhiều dữ liệu và tính toán hơn, nhưng trần của nó không do ai đặt sẵn. Đó là bài học đắng ở quy mô một sổ tay,
và cũng là lý do chương nói học sâu đổi điểm nghẽn thiết kế đặc trưng để lấy một điểm nghẽn tính toán mới.

## 2. Chi 10 % phép toán thêm vào đâu?

Trong mục định nghĩa hệ thống học máy, chương nêu một bài toán kinh tế:

In [ ]:
PHAN_TRAM = sach(10, trich="A 10 percent gain in cost efficiency can fund about 10 percent more useful operations")
HE_SO = sach(1.1, trich=r"governed by \(\alpha \log(1.1)\)")
assert math.isclose(HE_SO, 1 + PHAN_TRAM / 100)
print(f"{PHAN_TRAM} % hiệu quả chi phí hơn → khoảng {PHAN_TRAM} % phép toán hữu ích hơn, cùng ngân sách")
print(f"nếu sai số ∝ D^(−α), thêm {PHAN_TRAM} % dữ liệu nhân sai số với {HE_SO}^(−α), "
      f"tức giảm tương đối 1 − {HE_SO}^(−α) ≈ α·ln({HE_SO})")

Chương nói phần phép toán thêm đó có thể chi vào nhiều dữ liệu hơn, nhiều lượt huấn luyện hơn, hay một mô hình
lớn hơn, và lựa chọn tuỳ vào độ co giãn của đường cong học tập của chính tải công việc (workload) đó. Việc của
kỹ sư là ước lượng độ co giãn ấy cho hệ thống đang có.

Hãy làm đúng việc đó. Điểm gốc: mạng 784-64-10, 4,000 ảnh, 4 epoch. Rồi nhân đôi dần từng núm một, giữ hai núm
kia: số ảnh, số epoch, độ rộng lớp ẩn. Cả ba núm đều tăng tổng số phép toán huấn luyện gần như tỉ lệ thuận, theo
xấp xỉ huấn luyện dày đặc $O \approx 6 \cdot P \cdot \text{số lượt trình bày mẫu}$ (sổ tay 01). Khớp
$\text{sai số} \propto O^{-\alpha}$ trên mỗi hướng cho ta một $\alpha$ cho hướng đó.

**Dự đoán:** hướng nào có $\alpha$ lớn nhất ở điểm gốc này?

In [ ]:
GOC = {"n": 4_000, "epoch": 4, "rong": 64}
NUM = {"n": ("thêm dữ liệu", [1_000, 2_000, 4_000, 8_000]),
       "epoch": ("thêm lượt huấn luyện", [1, 2, 4, 8]),
       "rong": ("mô hình rộng hơn", [16, 32, 64, 128])}

t0 = time.perf_counter()
duong = {}
for num, (ten, gia_tri) in NUM.items():
    O, sai = [], []
    for v in gia_tri:
        cfg = dict(GOC, **{num: v})
        m, _ = huan_luyen(lambda: mlp((cfg["rong"],)), X[:cfg["n"]], y[:cfg["n"]], cfg["epoch"])
        O.append(6 * so_tham_so(m) * cfg["n"] * cfg["epoch"])
        sai.append(1 - do_chinh_xac(m, X_kt, y_kt))
    duong[num] = (np.array(O), np.array(sai))
do_tren_may("thời gian cho 12 lần huấn luyện", time.perf_counter() - t0, "s")

bang = []
for num, (O, sai) in duong.items():
    alpha = -np.polyfit(np.log(O), np.log(sai), 1)[0]
    bang.append({"hướng": NUM[num][0], "α (đo)": alpha,
                 f"giảm sai số khi thêm {PHAN_TRAM} % phép toán (%)": 100 * (1 - HE_SO ** -alpha),
                 "xấp xỉ α·ln(1.1) (%)": 100 * alpha * math.log(HE_SO),
                 "điểm phần trăm mỗi 10× phép toán": 100 * (sai[0] - sai[-1]) / math.log10(O[-1] / O[0])})
    do_tren_may(f"α theo hướng {NUM[num][0]}", alpha, "")
bang = pd.DataFrame(bang)
print(bang.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for (num, (O, sai)), mau, kieu in zip(duong.items(), MAU, ("o-", "s--", "^:")):
    ax.loglog(O / 1e9, 100 * sai, kieu, color=mau, lw=2, ms=6, label=NUM[num][0])
O_goc = 6 * so_tham_so(mlp((GOC["rong"],))) * GOC["n"] * GOC["epoch"]
ax.annotate(f"điểm gốc: {GOC['n']:,} ảnh, {GOC['epoch']} epoch, lớp ẩn {GOC['rong']}",
            (O_goc / 1e9, 100 * duong["n"][1][NUM["n"][1].index(GOC["n"])]), xytext=(12, 22),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
tat_ca_O = np.concatenate([O for O, _ in duong.values()]) / 1e9
tat_ca_sai = 100 * np.concatenate([sai for _, sai in duong.values()])
ax.minorticks_off()
ax.set_xticks(np.unique(tat_ca_O.round(2)), [f"{v:.2g}" for v in np.unique(tat_ca_O.round(2))])
moc_y = [v for v in (5, 6, 8, 10, 12, 15, 20, 25, 30) if tat_ca_sai.min() * 0.9 <= v <= tat_ca_sai.max() * 1.1]
ax.set_yticks(moc_y, [str(v) for v in moc_y])
ax.set_xlabel("GFLOP huấn luyện, xấp xỉ 6·P·số lượt (thang log)")
ax.set_ylabel("sai số kiểm tra (%, thang log)")
ax.set_title("Ba hướng chi thêm phép toán từ cùng một điểm (đo trên máy này)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Ba điều cần đọc cùng con số.

* $\alpha$ là độ dốc **tại điểm gốc này**, của tải công việc này. Nó là một ước lượng cục bộ, không phải hằng số
  của học máy: một điểm gốc khác, một mô hình khác hay một tập dữ liệu khác cho một $\alpha$ khác. Vì vậy chương
  nói khoản lợi từ thêm dữ liệu do $\alpha\log(1.1)$ quyết định chứ không do một tỉ lệ phần trăm phổ quát nào.
* Các hướng không cùng giá. Thêm lượt huấn luyện hay mở rộng mô hình chỉ tốn phép toán. Thêm dữ liệu còn tốn
  nhãn, một khoản chi nằm ngoài phép đếm FLOP.
* Cột cuối là lợi tức trên tính toán (return on compute, RoC) của sổ tay 01, đo theo từng hướng: bao nhiêu điểm
  độ chính xác cho mỗi lần nhân 10 số phép toán.

## 3. Ba chiều của hiệu quả, trên phương trình chi phí

Chương viết chi phí tính toán thành một tỉ số bậc nhất (phương trình 2):

$$\text{Compute Cost} \approx \frac{O_\text{total}}{\text{Useful Operations per Dollar}}$$

với $O_\text{total}$ là tổng số phép toán qua mọi lượt trình bày mẫu. Mỗi trục D·A·M cải thiện tỉ số này theo một
cách: dữ liệu tốt hơn giảm số lượt trình bày cần để đạt một mức chất lượng, thuật toán hiệu quả hơn giảm số phép
toán cần cho việc học, còn phần cứng hiệu quả hơn tăng số phép toán hữu ích mỗi đô la. Ba chiều hiệu quả của
chương, chọn lọc dữ liệu, hiệu quả thuật toán và hiệu quả tính toán, là ba cách đó.

### 3a. Chọn lọc dữ liệu: cho mô hình xem những ảnh nó còn phân vân

Thí nghiệm: có 10,000 ảnh chưa gán nhãn và một ngân sách nhãn. Cách thứ nhất gán nhãn ngẫu nhiên. Cách thứ hai
là lấy mẫu theo độ bất định: gán nhãn 300 ảnh ngẫu nhiên, huấn luyện một mô hình mồi, cho nó chấm mọi ảnh còn lại,
rồi gán nhãn những ảnh mà hai lớp nó tin nhất sát nhau nhất. Cả hai cách huấn luyện mô hình cuối 10 epoch, nên số
lượt trình bày tỉ lệ với số nhãn.

**Dự đoán:** cách thứ hai cần bao nhiêu nhãn để đạt độ chính xác mà cách thứ nhất đạt với 3,000 nhãn?

In [ ]:
NGAN_SACH = [500, 1_000, 1_500, 2_000, 3_000]
MOI, EPOCH_CUOI = 300, 10
kien_truc = lambda: mlp((128, 64))       # noqa: E731
P_mlp = so_tham_so(kien_truc())

rng = np.random.default_rng(SEED)
mo_dau = rng.choice(len(X), MOI, replace=False)
con_lai = np.setdiff1d(np.arange(len(X)), mo_dau)
moi, _ = huan_luyen(kien_truc, X[mo_dau], y[mo_dau], 30)
with torch.no_grad():
    hai_lop = F.softmax(moi(X[con_lai]), 1).topk(2, 1).values
bien = (hai_lop[:, 0] - hai_lop[:, 1]).numpy()           # càng nhỏ, mô hình càng phân vân
thu_tu = {"ngẫu nhiên": rng.permutation(con_lai), "theo độ bất định": con_lai[np.argsort(bien)]}

t0 = time.perf_counter()
chon_loc = {ten: [] for ten in thu_tu}
for k in NGAN_SACH:
    for ten, nguon_anh in thu_tu.items():
        chi_so = np.concatenate([mo_dau, nguon_anh[:k - MOI]])
        m, _ = huan_luyen(kien_truc, X[chi_so], y[chi_so], EPOCH_CUOI)
        chon_loc[ten].append(do_chinh_xac(m, X_kt, y_kt))
do_tren_may("thời gian cho 10 lần huấn luyện", time.perf_counter() - t0, "s")

muc_tieu = chon_loc["ngẫu nhiên"][-1]
dat = [k for k, a in zip(NGAN_SACH, chon_loc["theo độ bất định"]) if a >= muc_tieu]
do_tren_may(f"độ chính xác của cách ngẫu nhiên với {NGAN_SACH[-1]:,} nhãn", 100 * muc_tieu, "%")
O_mo_dau = 6 * P_mlp * MOI * 30 + 2 * P_mlp * len(con_lai)   # huấn luyện mô hình mồi + chấm phần còn lại
if dat:
    k = dat[0]
    do_tren_may("số nhãn nhỏ nhất trong dải đo mà cách theo độ bất định đạt mức đó", k, "nhãn")
    do_tren_may("số nhãn, cách ngẫu nhiên chia cách theo độ bất định", NGAN_SACH[-1] / k, "×")
    O_ngau = 6 * P_mlp * NGAN_SACH[-1] * EPOCH_CUOI
    O_bat_dinh = 6 * P_mlp * k * EPOCH_CUOI + O_mo_dau
    do_tren_may("O_total của cách ngẫu nhiên chia cho O_total của cách theo độ bất định (đã tính cả mô hình mồi)",
                O_ngau / O_bat_dinh, "×")
else:
    print("trong dải đo, cách theo độ bất định chưa đạt mức của cách ngẫu nhiên")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for (ten, acc), mau, kieu in zip(chon_loc.items(), MAU, ("s--", "o-")):
    ax.plot(NGAN_SACH, 100 * np.array(acc), kieu, color=mau, lw=2, ms=6, label=ten)
ax.axhline(100 * muc_tieu, color="0.5", ls=":", lw=1)
ax.text(NGAN_SACH[0], 100 * muc_tieu + 0.25, f"ngẫu nhiên, {NGAN_SACH[-1]:,} nhãn", fontsize=8, color="0.3")
ax.set_xticks(NGAN_SACH, [f"{k:,}" for k in NGAN_SACH])
ax.set_xlabel("số ảnh được gán nhãn")
ax.set_ylabel("độ chính xác kiểm tra (%)")
ax.set_title("Chọn ảnh nào để gán nhãn (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Nếu cách thứ hai đạt cùng mức với ít nhãn hơn, nó giảm hai thứ một lúc: số nhãn phải trả tiền, và số lượt trình
bày trong $O_\text{total}$. Đó là ý của chương khi nói chọn lọc dữ liệu rút nhiều tín hiệu học hơn từ một số ví dụ
có hạn, và nhờ đó giảm số hạng số phép toán $O$ của định luật sắt.

Nhưng hai tỉ số in ở trên không nhất thiết bằng nhau, và hãy so chúng. Tỉ số $O_\text{total}$ đã trừ đi phần cách
thứ hai phải trả thêm: huấn luyện mô hình mồi 30 epoch và chấm gần 10,000 ảnh. Đó là một khoản phép toán cố định,
không co lại theo ngân sách nhãn, nên ở quy mô nhỏ như ở đây nó có thể ăn mất phần lớn khoản tiết kiệm về phép
toán, dù khoản tiết kiệm về nhãn vẫn còn nguyên. Chọn lọc dữ liệu cũng có giá, và giá đó phải được đếm.

### 3b. Hiệu quả thuật toán: một bộ tối ưu tốt hơn

Chương định nghĩa hiệu quả thuật toán là giảm nhu cầu tính toán nhờ thiết kế mô hình và cách huấn luyện tốt hơn.
Cùng mạng 784-128-64-10, cùng dữ liệu, chỉ đổi bộ tối ưu (optimizer): hạ gradient ngẫu nhiên (stochastic gradient
descent, SGD) với tốc độ học (learning rate) 0.05, và Adam với tốc độ học 0.001. Mục tiêu là độ chính xác SGD đạt
sau 6 epoch.

**Dự đoán:** Adam cần mấy epoch để đạt cùng mức?

In [ ]:
SO_EPOCH = 6
LR_SGD = 0.05
t0 = time.perf_counter()
duong_hoc = {f"SGD, tốc độ học {LR_SGD}": huan_luyen(kien_truc, X, y, SO_EPOCH, "sgd", LR_SGD,
                                                     X_danh_gia=X_kt, y_danh_gia=y_kt)[1],
             "Adam, tốc độ học 0.001": huan_luyen(kien_truc, X, y, SO_EPOCH, "adam", 1e-3,
                                                  X_danh_gia=X_kt, y_danh_gia=y_kt)[1]}
do_tren_may("thời gian cho hai lần huấn luyện", time.perf_counter() - t0, "s")
muc_tieu_sgd = duong_hoc[f"SGD, tốc độ học {LR_SGD}"][-1]
dat_adam = [e + 1 for e, a in enumerate(duong_hoc["Adam, tốc độ học 0.001"]) if a >= muc_tieu_sgd]
do_tren_may(f"độ chính xác SGD sau {SO_EPOCH} epoch", 100 * muc_tieu_sgd, "%")
if dat_adam:
    do_tren_may("số epoch Adam cần để đạt mức đó", dat_adam[0], "epoch")
    do_tren_may("tỉ số O_total, SGD chia Adam, tới cùng mức", SO_EPOCH / dat_adam[0], "×")
else:
    print(f"Adam chưa đạt mức đó trong {SO_EPOCH} epoch")

luot = np.arange(1, SO_EPOCH + 1) * len(X)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for (ten, acc), mau, kieu in zip(duong_hoc.items(), MAU, ("s--", "o-")):
    ax.plot(luot, 100 * np.array(acc), kieu, color=mau, lw=2, ms=6, label=ten)
ax.axhline(100 * muc_tieu_sgd, color="0.5", ls=":", lw=1, label=f"mức SGD sau {SO_EPOCH} epoch")
ax.set_xticks(luot, [f"{v // 1000}k" for v in luot])
ax.set_xlabel("số lượt trình bày mẫu (O_total tỉ lệ thuận)")
ax.set_ylabel("độ chính xác kiểm tra (%)")
ax.set_title("Cùng mạng, cùng dữ liệu, khác bộ tối ưu (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Mỗi bước của Adam tốn thêm vài phép tính cho mỗi tham số để cập nhật trạng thái của nó, rất nhỏ so với 6 FLOP
mỗi tham số cho mỗi ảnh nhân 32 ảnh của một batch, nên tỉ số trên gần đúng là tỉ số $O_\text{total}$. Lưu ý một
điều: so sánh này phụ thuộc tốc độ học đã chọn cho SGD. Một tốc độ học khác có thể thu hẹp hay nới rộng khoảng
cách. Thử thêm 2 cho bạn xem điều đó.

### 3c. Hiệu quả tính toán: cùng phép toán, cách chạy khác

Chiều thứ ba không đổi $O$ chút nào. Nó tăng số phép toán hữu ích mà cùng một máy làm được mỗi giây, và vì thuê
máy tính tiền theo giờ, cũng là mỗi đô la. Thí nghiệm: một mạng 784-128-64-10, huấn luyện nhanh 1 epoch, chấm
5,000 ảnh kiểm tra hai lần: một lần từng ảnh một, một lần cả khối trong một phép nhân ma trận (matrix
multiplication) cho mỗi tầng.

In [ ]:
mo_hinh, _ = huan_luyen(kien_truc, X, y, 1)


def tung_anh():
    with torch.no_grad():
        return torch.cat([mo_hinh(X_kt[i:i + 1]) for i in range(len(X_kt))])


def ca_khoi():
    with torch.no_grad():
        return mo_hinh(X_kt)


def trung_vi_thoi_gian(f, lap=5):
    lan = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        lan.append(time.perf_counter() - t0)
    return float(np.median(lan))


O_suy_luan = 2 * P_mlp * len(X_kt)   # xuôi: 2 FLOP mỗi tham số mỗi ảnh
t_tung, t_khoi = trung_vi_thoi_gian(tung_anh, 3), trung_vi_thoi_gian(ca_khoi)
print(f"cả hai cách cùng {O_suy_luan / 1e9:.2f} GFLOP; số dự đoán khác nhau giữa hai cách: "
      f"{int((tung_anh().argmax(1) != ca_khoi().argmax(1)).sum())}")
do_tren_may("từng ảnh một", O_suy_luan / t_tung / 1e9, "GFLOP/s")
do_tren_may("cả khối", O_suy_luan / t_khoi / 1e9, "GFLOP/s")
do_tren_may("số phép toán hữu ích mỗi giây, cả khối chia từng ảnh", t_tung / t_khoi, "×")

Hai cách làm cùng số phép toán, và dòng in đầu tiên cho biết chúng có trả cùng câu trả lời không. Khác nhau là
cách chạy: chạy cả khối, mỗi trọng số được nạp vào bộ nhớ đệm (cache) một lần rồi dùng cho cả 5,000 ảnh, và các
đơn vị vector của vi xử lý luôn có việc. Chạy từng ảnh, cả ma trận trọng số phải đi qua bộ nhớ đệm 5,000 lần cho
những phép nhân nhỏ, và mỗi lần gọi còn trả thêm chi phí khởi động. Đó là ý của chương khi nói hiệu quả tính toán
khớp logic thuật toán với vật lý của máy. Sổ tay 01 đo đúng hiện tượng này trên một tầng tuyến tính.

Tóm lại phần 3, ba chiều chạm ba chỗ khác nhau của phương trình 2:

| chiều | thứ nó giảm hay tăng | thí nghiệm ở trên |
|---|---|---|
| chọn lọc dữ liệu | số lượt trình bày cần cho một mức chất lượng, trong $O_\text{total}$ | 3a, lấy mẫu theo độ bất định |
| hiệu quả thuật toán | phép toán cần để học tới một mức chất lượng, trong $O_\text{total}$ | 3b, đổi bộ tối ưu |
| hiệu quả tính toán | phép toán hữu ích mỗi đô la, ở mẫu số | 3c, chạy cả khối |

Sổ tay không đo chúng khi ghép lại. Hai chiều đầu cùng tác động lên số lượt trình bày, nên lợi ích của chúng có
thể chồng lên nhau, không nhất thiết nhân với nhau.

## 4. Các nhịp tăng trưởng

### 44.5× trong 7 năm

Theo chương, từ 2012 đến 2019, lượng tính toán cần để huấn luyện một mạng đạt mức AlexNet trên ImageNet giảm
khoảng 44.5×, tức giảm một nửa sau khoảng mỗi 15 tháng.

In [ ]:
NAM_DAU = sach(2012, trich="Between 2012 and 2019, computational resources needed to train a neural network")
NAM_CUOI = sach(2019, trich="Between 2012 and 2019, computational resources needed to train a neural network")
GIAM = sach(44.5, nguon="2012→2019 giảm 44,5× lượng tính toán cho cùng mức AlexNet")
thang = 12 * (NAM_CUOI - NAM_DAU)     # giả định: hai mốc năm cách nhau đúng 7 năm
nua_ky = thang / math.log2(GIAM)
print(f"{thang} tháng, {math.log2(GIAM):.2f} lần giảm một nửa")
theo_sach("thời gian giảm một nửa (tháng)", nua_ky, sach=15, nguon="giảm một nửa mỗi 15 tháng", tol=0.5)

### 3.4 tháng, và khoảng cách hệ thống

Cùng lúc đó, lượng tính toán dùng cho những lần huấn luyện tiên phong đã công bố tăng theo một đường khớp gấp
đôi sau khoảng mỗi 3.4 tháng. Chương so nó với nhịp phần cứng: mật độ transistor gấp đôi sau khoảng hai năm, theo
định luật Moore. Khoảng cách giữa hai nhịp đó là khoảng cách hệ thống.

Chương viết nhịp của định luật Moore bằng chữ, nên sổ tay ghi nó là một giả định: 24 tháng.

**Dự đoán:** trong cùng 84 tháng, nhịp 3.4 tháng cho bao nhiêu bậc độ lớn?

In [ ]:
GAP_DOI = sach(3.4, nguon="tính toán huấn luyện tăng gấp đôi mỗi 3,4 tháng")
MOORE = 24     # giả định: "roughly every two years", như chương viết
nhip = {"lượng tính toán huấn luyện tiên phong (gấp đôi mỗi 3.4 tháng)": GAP_DOI,
        "mật độ transistor (gấp đôi mỗi 24 tháng, giả định)": MOORE,
        f"hiệu quả thuật toán (gấp đôi mỗi {nua_ky:.1f} tháng)": nua_ky}
for ten, T in nhip.items():
    print(f"{ten}: ×{2 ** (12 / T):.2f} mỗi năm, ×{2 ** (thang / T):,.3g} sau {thang} tháng")

cung = 2 ** (12 / MOORE) * 2 ** (12 / nua_ky)
cau = 2 ** (12 / GAP_DOI)
print(f"phần cứng và thuật toán cộng lại: ×{cung:.2f} mỗi năm; nhu cầu: ×{cau:.2f} mỗi năm")
print(f"phần còn lại, ×{cau / cung:.2f} mỗi năm, phải đến từ nhiều chip hơn, chạy lâu hơn, tốn nhiều tiền hơn")

BAC = sach(7, nguon="khoảng cách tổng 7 bậc độ lớn")
ALEXNET = sach(2012, trich="Deep Learning (2012)")
GPT4 = sach(2023, trich="Modern Deep Learning (2023)")
nhip_dau_cuoi = 12 * (GPT4 - ALEXNET) / math.log2(10 ** BAC)
print(f"nhịp 3.4 tháng trong {thang} tháng: {thang / GAP_DOI * math.log10(2):.1f} bậc độ lớn")
print(f"{BAC} bậc độ lớn giữa hai mốc AlexNet ({ALEXNET}) và GPT-4 ({GPT4}): tương đương gấp đôi mỗi "
      f"{nhip_dau_cuoi:.1f} tháng")

nam = np.linspace(0, thang, 85)
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for (ten, T), mau, kieu in zip(nhip.items(), (MAU[1], MAU[0], MAU[2]), ("-", "--", ":")):
    ax.semilogy(NAM_DAU + nam / 12, 2 ** (nam / T), kieu, color=mau, lw=2.2, label=ten)
ax.plot([NAM_CUOI], [GIAM], "o", color=MAU[2], ms=7)
ax.annotate(f"sách: {GIAM}× (2012→2019)", (NAM_CUOI, GIAM), xytext=(-120, 18), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate("", xy=(NAM_CUOI, 2 ** (thang / GAP_DOI)), xytext=(NAM_CUOI, 2 ** (thang / MOORE)),
            arrowprops=dict(arrowstyle="<->", color="0.3", lw=1))
ax.text(NAM_CUOI - 0.15, math.sqrt(2 ** (thang / GAP_DOI) * 2 ** (thang / MOORE)), "khoảng cách\nhệ thống",
        ha="right", va="center", fontsize=8, color="0.25")
ax.set_xlabel("năm")
ax.set_ylabel("bội số so với 2012 (thang log)")
ax.set_title("Ba nhịp, cùng một trục")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Hai phép tính cuối không khớp nhau, và chương nói vì sao: đường khớp 3.4 tháng là một xu hướng tổng hợp trên các
lần huấn luyện tiên phong đã công bố, còn 7 bậc độ lớn là tỉ số giữa hai mô hình mốc. Hai đại lượng khác nhau thì
nhịp suy ra từ chúng cũng khác, và sổ tay không ép chúng bằng nhau.

Còn hai con số 44.5× và 7 bậc độ lớn thì nghe như mâu thuẫn: một bên hiệu quả tăng, bên kia nhu cầu tăng. Chương
gỡ nó bằng việc chỉ ra hai phép đo giữ cố định hai thứ khác nhau. Phép đo thứ nhất giữ cố định mức năng lực và hỏi
cần bao nhiêu tính toán. Phép đo thứ hai để mức năng lực tăng lên và ghi lại ngân sách huấn luyện.

Góc nhìn hệ thống 1.4 gọi đó là nghịch lý Jevons của học máy: khi một tài nguyên được dùng hiệu quả hơn, tổng
lượng tiêu thụ thường tăng chứ không giảm, vì phần tiết kiệm được đem chi vào mô hình lớn hơn và dữ liệu rộng
hơn. Dòng "phần còn lại" ở trên cho thấy cái giá của vòng lặp đó: hiệu quả thuật toán và phần cứng cộng lại vẫn
không theo kịp nhu cầu, và phần chênh được trả bằng quy mô.

## Thử thêm

1. Ở phần 1, đổi `lambda: mlp((128, 64))` của cách điểm ảnh thô thành `lambda: mlp((16,))`. Chỗ các đường cắt
   nhau, nếu có, dịch về đâu? Một mạng nhỏ có còn vượt được đặc trưng thủ công không?
2. Ở phần 3b, đổi `LR_SGD = 0.05` thành `LR_SGD = 0.2`, rồi thành `0.01`. Khoảng cách với Adam thay đổi thế
   nào? Một bộ tối ưu "tốt hơn" được đo so với một cấu hình cụ thể của bộ tối ưu kia.
3. Ở phần 2, đổi điểm gốc thành `GOC = {"n": 1_000, "epoch": 2, "rong": 32}`, và trong `NUM` đổi danh sách số ảnh
   thành `[250, 500, 1_000, 2_000]` để điểm gốc nằm trong dải. Thứ tự của ba $\alpha$ có giữ nguyên không?

## Tóm lại

* Đặc trưng do người thiết kế mang sẵn hiểu biết của con người, nên có lợi khi dữ liệu ít; một mô hình tự học
  đặc trưng từ điểm ảnh cần nhiều dữ liệu và tính toán hơn, nhưng trần của nó không do ai đặt sẵn.
* Thêm 10 % phép toán vào đâu là câu hỏi về độ co giãn $\alpha$ của chính tải công việc đó. $\alpha$ đo được tại
  một điểm, theo một hướng, và giảm tương đối của sai số xấp xỉ $\alpha\ln(1.1)$.
* Phương trình 2 đặt ba chiều hiệu quả vào ba chỗ: chọn lọc dữ liệu và hiệu quả thuật toán giảm $O_\text{total}$,
  còn hiệu quả tính toán tăng phép toán hữu ích mỗi đô la.
* 44.5× trong 84 tháng là giảm một nửa sau mỗi 15.3 tháng; nhịp 3.4 tháng nhanh hơn nhiều so với cả phần cứng lẫn
  thuật toán cộng lại, và khoảng chênh đó được trả bằng quy mô.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nhập môn hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch01-introduction/), dựng từ chương
[*Introduction*](https://mlsysbook.ai/vol1/introduction/introduction.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.